# Docling PDF Extraction Pipeline — With Boundary Tags

This notebook extends the base extraction notebook with **HTML boundary markers** around every element.
Each element is wrapped in `<!-- BOUNDARY_START ... -->` / `<!-- BOUNDARY_END -->` tags that carry
structured metadata (type, id, page, breadcrumbs, etc.) so downstream consumers — RAG chunkers,
retrieval pipelines, QA systems — can locate and slice any element precisely without re-parsing.

```
<!-- BOUNDARY_START type="table" id="p5_table_1" page="5" rows="8" columns="4" breadcrumbs="Results > Revenue" -->
| Col A | Col B |
| ----- | ----- |
| 1     | 2     |
<!-- BOUNDARY_END type="table" id="p5_table_1" -->
```

| Stage | What happens |
|---|---|
| **1. Imports** | Standard + Docling + OpenAI |
| **2. Config** | Paths, model, image scale |
| **3. Dirs** | Output directory scaffold |
| **4. Boundary helpers** | `create_boundary_start/end`, `wrap_with_boundaries` |
| **5. ID generator** | Deterministic `p{page}_{type}_{counter}` IDs |
| **6. Converter** | Docling `PdfPipelineOptions` + `TableFormerMode.ACCURATE` |
| **7. Convert PDF** | Docling runs layout analysis |
| **8. Group by page** | Items bucketed into `pages_items` dict |
| **9. AI helpers** | `describe_table_with_ai`, `analyse_figure_with_gpt4o` |
| **10. Item processors** | `process_header`, `process_text`, `process_list`, `process_image`, `process_table` |
| **11. Main loop** | Per-page dispatch with caption detection + skip logic |
| **12. Assembly** | Full document markdown + metadata JSON |

## Cell 1 – Imports

In [10]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================
# We store API keys (OpenAI, Anthropic, Google etc.) in a .env file
# so they are NEVER hard-coded in our notebooks or committed to Git.
#
# The .env file lives in the project root and looks like this:
#   OPENAI_API_KEY=sk-...
#   ANTHROPIC_API_KEY=sk-ant-...
#   GOOGLE_API_KEY=AI...
#
# load_dotenv() reads that file and injects those values into
# os.environ so that LangChain (and any other library) can find them
# automatically without us passing the key explicitly.
#
# Returns True if the .env file was found and loaded successfully.

from dotenv import load_dotenv

load_dotenv()

True

In [11]:
# ── Standard library ──────────────────────────────────────────────────────
import base64
import io
import json
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# ── Docling ───────────────────────────────────────────────────────────────
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.datamodel.base_models import InputFormat
from docling.datamodel.document import (
    TableItem,         # ML-parsed table
    PictureItem,       # figure / chart / image
    TextItem,          # paragraph or code block
    SectionHeaderItem, # heading
    ListItem,          # bullet / numbered list entry
)

# ── Third-party ───────────────────────────────────────────────────────────
from openai import OpenAI   # GPT-4o for table + figure analysis
import pandas as pd          # DataFrame → CSV / Markdown table

## Cell 2 – Global configuration

In [12]:
# ── Paths ──────────────────────────────────────────────────────────────────
pdf_path = Path(
    "/Users/akellaprudhvi/mystuff/Course/GenAI-Course-Modules/"
    "Module_4_notebooks/docs/AI-Enablers-Adopters-research-report.pdf"
)

# Root directory — all extracted artefacts are written under here
OUTPUT_DIR = Path("extracted_docs_bounded")

# ── OpenAI ────────────────────────────────────────────────────────────────
# Must be a vision-capable model for figure analysis
OPENAI_MODEL = "gpt-4o"

# ── Image scale ───────────────────────────────────────────────────────────
# 3.0 × native PDF resolution → ~216 DPI; legible chart text for GPT-4o Vision
IMAGE_SCALE = 3.0

## Cell 3 – Output directory scaffold

In [13]:
# One subdirectory per PDF (named after the file stem) — safe to run
# multiple PDFs into the same OUTPUT_DIR without filename collisions
doc_output_dir = OUTPUT_DIR / pdf_path.stem

# pages/   → one boundary-tagged .md file per page
# figures/ → PNG images for every PictureItem captured by Docling
(doc_output_dir / "pages").mkdir(parents=True, exist_ok=True)
(doc_output_dir / "figures").mkdir(parents=True, exist_ok=True)

print(f"Output root: {doc_output_dir.resolve()}")

Output root: /Users/akellaprudhvi/mystuff/Course/GenAI-Course-Modules/Module_4_notebooks/layout_identification_extraction/extracted_docs_bounded/AI-Enablers-Adopters-research-report


## Cell 4 – Boundary marker helpers

Every extracted element is wrapped in HTML comment tags:
```
<!-- BOUNDARY_START type="paragraph" id="p3_text_2" page="3" char_count="412" breadcrumbs="Intro > Background" -->
The actual paragraph text goes here.
<!-- BOUNDARY_END type="paragraph" id="p3_text_2" -->
```

Benefits:
- **RAG chunkers** can split on clean semantic boundaries instead of arbitrary token windows
- **Retrieval pipelines** can filter by type (`"give me all tables on page 5"`)
- **Document structure** (breadcrumb trail) is preserved inside each chunk

In [14]:
def create_boundary_start(item_type: str, item_id: str, page: int, **attrs) -> str:
    """
    Build an opening boundary comment tag.

    Args:
        item_type : element category e.g. 'paragraph', 'table', 'image'
        item_id   : unique element ID e.g. 'p3_text_2'
        page      : 1-based page number
        **attrs   : extra key=value pairs to embed (rows, columns, language, etc.)

    Returns:
        HTML comment string:
        <!-- BOUNDARY_START type="table" id="p5_table_1" page="5" rows="8" -->
    """
    attr_str = f'type="{item_type}" id="{item_id}" page="{page}"'
    for key, value in attrs.items():
        attr_str += f' {key}="{value}"'
    return f"<!-- BOUNDARY_START {attr_str} -->"


def create_boundary_end(item_type: str, item_id: str) -> str:
    """
    Build a closing boundary comment tag.

    The matching END tag lets parsers verify completeness and handle
    truncated outputs gracefully.

    Returns:
        e.g. <!-- BOUNDARY_END type="table" id="p5_table_1" -->
    """
    return f'<!-- BOUNDARY_END type="{item_type}" id="{item_id}" -->'


def wrap_with_boundaries(content: str, item_type: str, item_id: str,
                         page: int, **attrs) -> str:
    """
    Sandwich content between START and END boundary markers.

    None-valued attrs are filtered out before rendering so the tag stays
    clean — e.g. caption=None won't appear as caption="None" in the output.

    Returns:
        Multi-line string: START tag + content + END tag
    """
    # Drop None attrs — they carry no information and clutter the tag
    filtered_attrs = {k: v for k, v in attrs.items() if v is not None}
    start = create_boundary_start(item_type, item_id, page, **filtered_attrs)
    end   = create_boundary_end(item_type, item_id)
    return f"{start}\n{content}\n{end}"


print("Boundary helpers ready.")

# ── Quick smoke test ──────────────────────────────────────────────────────
print()
print(wrap_with_boundaries(
    "Sample paragraph text.",
    item_type="paragraph", item_id="p1_text_1", page=1,
    char_count=23, breadcrumbs="Introduction"
))

Boundary helpers ready.

<!-- BOUNDARY_START type="paragraph" id="p1_text_1" page="1" char_count="23" breadcrumbs="Introduction" -->
Sample paragraph text.
<!-- BOUNDARY_END type="paragraph" id="p1_text_1" -->


## Cell 5 – Unique ID generator

Each element gets a deterministic, human-readable ID: `p{page}_{type}_{counter}`

- `p1_header_1` — first heading on page 1
- `p3_text_2` — second paragraph on page 3
- `p7_image_1` — first figure on page 7

Short, sortable, meaningful when reading raw Markdown — and IDs can be
reset per-document so they never bleed across multiple PDFs in a batch.

In [15]:
# Module-level counter: key = "p{page}_{type}", value = current count
_id_counters: dict = defaultdict(int)


def generate_unique_id(page: int, item_type: str) -> str:
    """
    Generate a unique, human-readable element ID.

    Uses a per-page-per-type counter so each page's elements are numbered
    independently — easy to cross-reference against the source PDF.

    Args:
        page      : 1-based page number
        item_type : short category label e.g. 'text', 'table', 'image'

    Returns:
        e.g. 'p3_text_2' (second text block on page 3)
    """
    key = f"p{page}_{item_type}"
    _id_counters[key] += 1
    return f"{key}_{_id_counters[key]}"


def reset_id_counters():
    """
    Reset all counters to zero.

    Must be called at the start of each PDF in a batch so IDs from a
    previous document don't leak into the next one.
    """
    global _id_counters
    _id_counters = defaultdict(int)


print("ID generator ready.")

# ── Quick smoke test ──────────────────────────────────────────────────────
print(generate_unique_id(3, "text"))   # → p3_text_1
print(generate_unique_id(3, "text"))   # → p3_text_2
print(generate_unique_id(3, "table"))  # → p3_table_1
reset_id_counters()
print(generate_unique_id(3, "text"))   # → p3_text_1 (reset worked)
reset_id_counters()                    # reset before actual processing

ID generator ready.
p3_text_1
p3_text_2
p3_table_1
p3_text_1


## Cell 6 – Docling converter

In [16]:
pipeline_options = PdfPipelineOptions()

# Render figures and table crops at 3× resolution
pipeline_options.images_scale = IMAGE_SCALE

# Save figure PNGs alongside the DoclingDocument
pipeline_options.generate_picture_images = True

# Save table PNGs as a fallback for when text export fails
pipeline_options.generate_table_images = True

# Skip OCR — assumes the PDF has embedded selectable text.
# Set do_ocr=True for scanned / image-only PDFs.
pipeline_options.do_ocr = False

# Enable TableFormer — Docling's ML-based table structure detector
pipeline_options.do_table_structure = True

# ACCURATE mode = full TableFormer model → best row/col detection.
# Use TableFormerMode.FAST for quicker but lower-quality extraction.
pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

print("Converter ready.")

Converter ready.


## Cell 7 – Convert the PDF

In [17]:
# Heavy step — Docling runs layout analysis + TableFormer.
# Typical throughput: ~1–3 pages/sec depending on table density.
print(f"Converting: {pdf_path.name} …")
conv_result = converter.convert(pdf_path)
doc = conv_result.document

print(f"Done. Pages detected: {len(doc.pages)}")

Converting: AI-Enablers-Adopters-research-report.pdf …
Done. Pages detected: 7


## Cell 8 – Group document items by page

In [18]:
# doc.iterate_items() yields (item, level) pairs in reading order.
# We bucket them by page number so the main loop can process page-by-page
# and correctly look one item ahead for caption detection.

# pages_items: { page_no → [ { 'item': ..., 'level': int }, ... ] }
pages_items: dict = defaultdict(list)

for item, level in doc.iterate_items():
    # prov (provenance) holds the bounding box + page number.
    # Items without provenance are structural metadata — skip them.
    if not item.prov:
        continue
    page_num = item.prov[0].page_no
    pages_items[page_num].append({"item": item, "level": level})

total_items = sum(len(v) for v in pages_items.values())
print(f"Collected {total_items} items across {len(pages_items)} pages.")

Collected 102 items across 7 pages.


## Cell 9 – AI description helpers

Two helpers for GPT-4o analysis:
- **Tables** are sent as Markdown text — cheaper than Vision and lossless for structured data
- **Figures** are base64-encoded and sent as inline data URLs — no hosting required

In [19]:
# Initialise the OpenAI client (reads OPENAI_API_KEY from environment)
openai_client = OpenAI()


def describe_table_with_ai(table_text: str, caption: Optional[str] = None) -> str:
    """
    Ask GPT-4o to describe a table's purpose, structure, and key takeaways.

    Sends as Markdown text — cheaper than Vision and works well for
    structured data. The optional caption gives the model context about
    what the table represents (e.g. 'Exhibit 3: Revenue by Region FY2024').

    Returns:
        AI-generated description string, or error message on failure.
    """
    try:
        prompt = "Analyze this table. Describe its purpose, structure, and key information concisely."
        if caption:
            prompt += f"\n\nCaption: {caption}"
        prompt += f"\n\nTable:\n{table_text}"

        response = openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150  # short descriptions keep cost manageable at scale
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        # Non-fatal — return placeholder so the table is still written
        return f"AI description failed: {str(e)}"


def analyse_figure_with_gpt4o(image_path: Path, caption: Optional[str] = None) -> str:
    """
    Ask GPT-4o Vision to analyse a chart or figure image.

    The image is base64-encoded and sent as a data URL — self-contained
    in the API request, no hosting or signed URLs required.

    Args:
        image_path : Path to the saved PNG on disk
        caption    : optional caption for added context

    Returns:
        AI-generated analysis string, or error message on failure.
    """
    try:
        # Read the PNG from disk and encode to base64
        with open(image_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")

        # Ask for type classification AND analytical insight
        prompt = "Analyze this visual. Is it a Chart, Diagram, or Data Table? "
        prompt += "Describe axes, trends, and key insights concisely."
        if caption:
            prompt += f"\n\nCaption/Context: {caption}"

        response = openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    # data URL embeds the image bytes directly in the request
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}}
                ]
            }],
            max_tokens=200
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"AI description failed: {str(e)}"


print("AI helpers ready.")

AI helpers ready.


## Cell 10 – Item processor functions

Each processor handles one Docling item type and returns a **boundary-wrapped Markdown string**.
They all follow the same contract:
- Accept the Docling item + page/context info
- Call `generate_unique_id()` for a stable element ID
- Return a `wrap_with_boundaries(...)` string (or `""` to signal skip)

Processors are intentionally pure functions (no side effects except `_id_counters`)
so they are easy to test and reason about independently.

In [20]:
# ─────────────────────────────────────────────────────────────────────────
# PROCESSOR: Section headers
# ─────────────────────────────────────────────────────────────────────────

def process_header(item: SectionHeaderItem, page: int, level: int,
                   breadcrumbs: List[str]) -> Tuple[str, List[str]]:
    """
    Convert a section header to a Markdown heading and update the breadcrumb trail.

    BREADCRUMB LOGIC
    Breadcrumbs track the current document hierarchy position so that
    every subsequent element knows which section it belongs to — critical
    for RAG because a chunk containing "Revenue grew 12%" is far more
    useful when its metadata includes "Results > Revenue > Q4 Performance".

    Truncation rule: when a level-N header is encountered, all breadcrumbs
    at level N and deeper are discarded and replaced with the new header,
    mirroring how a table of contents works.

    Returns:
        Tuple of (wrapped Markdown string, updated breadcrumbs list)
    """
    text = item.text.strip()

    # Trim breadcrumbs to parent level before appending the new header
    if len(breadcrumbs) >= level:
        breadcrumbs = breadcrumbs[:level - 1]
    breadcrumbs.append(text)

    item_id = generate_unique_id(page, "header")
    # +1 to level: level 1 → ## (# is reserved for the page heading)
    content = f"{'#' * (level + 1)} {text}"

    output = wrap_with_boundaries(
        content, "header", item_id, page,
        level=level,
        breadcrumbs=" > ".join(breadcrumbs)  # e.g. "Results > Revenue > Q4"
    )
    return output, breadcrumbs


# ─────────────────────────────────────────────────────────────────────────
# PROCESSOR: Body text / paragraphs
# ─────────────────────────────────────────────────────────────────────────

def process_text(item: TextItem, page: int, breadcrumbs: List[str]) -> str:
    """
    Wrap a regular text paragraph in boundary markers.

    char_count and word_count in the tag are useful for:
      - Chunking strategies that target specific token ranges
      - Filtering noise (very short fragments < 50 chars)

    Single-character items (OCR noise, stray punctuation) are skipped.

    Returns:
        Boundary-wrapped Markdown string, or "" if text is too short.
    """
    text = item.text.strip()

    # Skip single characters — usually OCR noise or stray punctuation
    if len(text) <= 1:
        return ""

    item_id = generate_unique_id(page, "text")

    return wrap_with_boundaries(
        text, "paragraph", item_id, page,
        char_count=len(text),
        word_count=len(text.split()),
        breadcrumbs=" > ".join(breadcrumbs)
    )


# ─────────────────────────────────────────────────────────────────────────
# PROCESSOR: List items
# ─────────────────────────────────────────────────────────────────────────

def process_list(item: ListItem, page: int, breadcrumbs: List[str]) -> str:
    """
    Format a list item with its bullet marker and wrap in boundary markers.

    Docling exposes the marker via item.enumeration — could be a bullet
    character ('-', '•') or a number ('1.', '2.'). Fallback to '-' ensures
    the output is always valid Markdown.

    Returns:
        Boundary-wrapped Markdown list item string.
    """
    # getattr with default handles items where enumeration is not set
    marker = getattr(item, "enumeration", "-")
    text   = item.text.strip()
    content = f"{marker} {text}"
    item_id = generate_unique_id(page, "list")

    return wrap_with_boundaries(
        content, "list", item_id, page,
        breadcrumbs=" > ".join(breadcrumbs)
    )


# ─────────────────────────────────────────────────────────────────────────
# PROCESSOR: Figures / PictureItems
# ─────────────────────────────────────────────────────────────────────────

def process_image(item: PictureItem, page: int, image_counter: int,
                  breadcrumbs: List[str],
                  next_item=None) -> Tuple[str, int]:
    """
    Extract a figure PNG, get a GPT-4o Vision description, and return
    boundary-wrapped Markdown with an embedded relative image link.

    CAPTION DETECTION
    The item immediately following a figure is checked for Docling's
    'caption' label. If found, the caption is included in both the
    boundary metadata and the AI prompt for richer context.

    IMAGE PATH
    The link uses `../figures/<filename>` so it resolves correctly when
    the .md file is opened from the pages/ subdirectory.

    Returns:
        Tuple of (boundary-wrapped Markdown string, updated image_counter).
        Returns ("", image_counter) if extraction fails.
    """
    # Detect caption: next item in stream with Docling label 'caption'
    caption = None
    if next_item and isinstance(next_item, TextItem):
        if getattr(next_item, "label", None) == "caption":
            caption = next_item.text.strip()

    # Filename convention: fig_p{page}_{counter}.png
    filename = f"fig_p{page}_{image_counter}.png"
    filepath = doc_output_dir / "figures" / filename

    try:
        # Docling 2.x: get_image(doc) returns a PIL.Image directly — no unpacking
        pil_img = item.get_image(doc)
        if pil_img is None:
            return "", image_counter

        # Persist the figure PNG to disk
        pil_img.save(filepath)

        # GPT-4o Vision analysis — pass caption for richer context
        ai_desc = analyse_figure_with_gpt4o(filepath, caption)

    except Exception as e:
        print(f"   WARNING: Figure extraction failed on page {page}: {e}")
        return "", image_counter

    item_id = generate_unique_id(page, "image")

    # Build content block:
    #   **Image**
    #   *Caption:* ...   (if found)
    #   ![filename](../figures/filename.png)   ← relative path from pages/
    #   *AI Analysis:* ...
    content_parts = ["**Image**"]
    if caption:
        content_parts.append(f"*Caption:* {caption}")
    # ../figures/ makes the link relative to the pages/ directory
    content_parts.append(f"![{filename}](../figures/{filename})")
    content_parts.append(f"*AI Analysis:* {ai_desc}")
    content = "\n".join(content_parts)

    output = wrap_with_boundaries(
        content, "image", item_id, page,
        filename=filename,
        has_caption="yes" if caption else "no",
        breadcrumbs=" > ".join(breadcrumbs)
    )
    # Increment counter — next figure on this page gets a different filename
    return output, image_counter + 1


# ─────────────────────────────────────────────────────────────────────────
# PROCESSOR: Tables
# ─────────────────────────────────────────────────────────────────────────

def process_table(item: TableItem, page: int, image_counter: int,
                  breadcrumbs: List[str],
                  next_item=None) -> Tuple[str, int]:
    """
    Process a table using a text-first, image-fallback strategy.

    STAGE 1 — Text export (preferred)
      Docling's TableFormer exports the table as a pandas DataFrame.
      If non-empty, it is converted to Markdown and sent to GPT-4o as text.
      Text export is: cheaper (no Vision call), lossless, directly searchable.

    STAGE 2 — Image fallback
      If text export fails (empty DataFrame, malformed table, rendering error)
      the table is rendered to a PNG and sent to GPT-4o Vision.
      Handles merged-cell tables and image-based tables in scanned PDFs.

    CAPTION DETECTION
      Uses text-pattern heuristics (starts with 'Exhibit', 'Table', 'Source:', etc.)
      because table captions in financial PDFs often appear ABOVE the table and
      Docling doesn't always assign them the 'caption' label.

    Returns:
        Tuple of (boundary-wrapped Markdown string, updated image_counter).
        Counter only increments when the image-fallback path is taken.
    """
    # Caption detection for tables uses text-pattern heuristics
    caption = None
    if next_item and isinstance(next_item, TextItem):
        text = next_item.text.strip()
        is_caption = (
            text.startswith(("Exhibit", "Figure", "Table", "Chart", "Source:")) or
            "Source:" in text or
            (len(text) < 200 and ":" in text)  # short colon-containing line
        )
        if is_caption:
            caption = text

    # ── Stage 1: Text export ──────────────────────────────────────────────
    text_table_valid = False
    df = None
    md_table = None

    try:
        df = item.export_to_dataframe()
        if not df.empty and len(df) > 0 and len(df.columns) > 0:
            md_table = df.to_markdown(index=False)  # requires tabulate
            # Very short 'tables' (<50 chars) are likely noise — skip them
            text_table_valid = len(md_table) > 50
    except Exception:
        # export_to_dataframe can raise on malformed tables — catch all
        text_table_valid = False

    if text_table_valid and md_table is not None:
        item_id     = generate_unique_id(page, "table")
        table_desc  = describe_table_with_ai(md_table, caption)

        content_parts = []
        if caption:
            content_parts.append(f"*Caption:* {caption}")
        content_parts.append(md_table)
        content_parts.append(f"\n*AI Analysis:* {table_desc}")
        content = "\n".join(content_parts)

        output = wrap_with_boundaries(
            content, "table", item_id, page,
            rows=len(df),
            columns=len(df.columns),
            has_caption="yes" if caption else "no",
            breadcrumbs=" > ".join(breadcrumbs)
        )
        # Counter NOT incremented — no image file was created
        return output, image_counter

    # ── Stage 2: Image fallback ───────────────────────────────────────────
    filename = f"fig_p{page}_{image_counter}.png"
    filepath = doc_output_dir / "figures" / filename

    try:
        pil_img = item.get_image(doc)   # Docling 2.x returns PIL.Image directly
        if pil_img is None:
            return "", image_counter

        pil_img.save(filepath)
        ai_desc = analyse_figure_with_gpt4o(filepath, caption)

    except Exception as e:
        print(f"   WARNING: Table image fallback failed on page {page}: {e}")
        return "", image_counter

    item_id = generate_unique_id(page, "image")  # labelled 'image' — it's a PNG

    content_parts = ["**Table/Chart**"]
    if caption:
        content_parts.append(f"*Caption:* {caption}")
    content_parts.append(f"![{filename}](../figures/{filename})")
    content_parts.append(f"*AI Analysis:* {ai_desc}")
    content = "\n".join(content_parts)

    output = wrap_with_boundaries(
        content, "image", item_id, page,
        filename=filename,
        has_caption="yes" if caption else "no",
        breadcrumbs=" > ".join(breadcrumbs)
    )
    # Counter IS incremented — a PNG was written to disk
    return output, image_counter + 1


print("All item processors ready.")

All item processors ready.


## Cell 11 – Main per-page processing loop

For each page:
1. Inject a breadcrumb context comment (carries section context across page breaks)
2. Walk every item in reading order, dispatching to the appropriate processor
3. Items identified as captions are added to `skip_indices` so they don't
   also appear as standalone paragraphs
4. Write the assembled boundary-tagged content to `pages/page_{N}.md`

In [21]:
# Reset ID counters before processing this document
reset_id_counters()

# ── Per-document accumulators ──────────────────────────────────────────────
global_breadcrumbs: List[str] = []  # persists across pages — section context carries over
all_pages_content: List[Tuple[int, str]] = []  # (page_no, markdown_text)
metadata_pages: List[dict] = []
total_images = 0
total_tables = 0

# ──────────────────────────────────────────────────────────────────────────
for page_num in sorted(pages_items.keys()):
    items = pages_items[page_num]

    page_outputs: List[str] = []
    page_image_count = 0
    page_table_count = 0
    image_counter    = 1    # resets per page: fig_p3_1, fig_p3_2, ...
    skip_indices: set = set()  # indices of items consumed as captions

    # Carry breadcrumb context from the previous page as an HTML comment
    # so readers know which section they're in even without prior pages
    if global_breadcrumbs:
        page_outputs.append(f"<!-- Context: {' > '.join(global_breadcrumbs)} -->")
    page_outputs.append(f"# Page {page_num}\n")

    for idx, entry in enumerate(items):
        # Skip items already consumed as captions for a preceding element
        if idx in skip_indices:
            continue

        item  = entry["item"]
        level = entry["level"]

        # Look one item ahead — needed for caption detection in image/table processors
        next_item = items[idx + 1]["item"] if idx + 1 < len(items) else None

        # ── Dispatch by item type ─────────────────────────────────────────

        if isinstance(item, SectionHeaderItem):
            output, global_breadcrumbs = process_header(
                item, page_num, level, global_breadcrumbs
            )
            page_outputs.append(output)

        elif isinstance(item, TextItem):
            output = process_text(item, page_num, global_breadcrumbs)
            if output:  # empty string = item was too short, skip
                page_outputs.append(output)

        elif isinstance(item, ListItem):
            output = process_list(item, page_num, global_breadcrumbs)
            page_outputs.append(output)

        elif isinstance(item, PictureItem):
            output, image_counter = process_image(
                item, page_num, image_counter, global_breadcrumbs, next_item
            )
            if output:
                page_outputs.append(output)
                page_image_count += 1
                # If next item was consumed as a caption, mark it for skipping
                # so it doesn't also appear as a standalone paragraph
                if next_item and isinstance(next_item, TextItem):
                    if getattr(next_item, "label", None) == "caption":
                        skip_indices.add(idx + 1)

        elif isinstance(item, TableItem):
            output, image_counter = process_table(
                item, page_num, image_counter, global_breadcrumbs, next_item
            )
            if output:
                page_outputs.append(output)
                page_table_count += 1
                # Same caption-skip logic as images
                if next_item and isinstance(next_item, TextItem):
                    text = next_item.text.strip()
                    is_cap = (
                        text.startswith(("Exhibit", "Figure", "Table", "Chart", "Source:")) or
                        "Source:" in text or
                        (len(text) < 200 and ":" in text)
                    )
                    if is_cap:
                        skip_indices.add(idx + 1)

    # ── Write per-page .md file ───────────────────────────────────────────
    page_text = "\n\n".join(page_outputs)
    page_file = doc_output_dir / "pages" / f"page_{page_num}.md"
    page_file.write_text(page_text, encoding="utf-8")

    all_pages_content.append((page_num, page_text))

    metadata_pages.append({
        "page": page_num,
        "file": page_file.name,
        "breadcrumbs": list(global_breadcrumbs),
        "images": page_image_count,
        "tables": page_table_count
    })

    total_images += page_image_count
    total_tables += page_table_count

    print(f"  ✓ Page {page_num:3d} — {page_image_count} images, {page_table_count} tables → {page_file.name}")

print(f"\nExtraction complete: {total_tables} tables, {total_images} images.")

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


  ✓ Page   1 — 2 images, 1 tables → page_1.md
  ✓ Page   2 — 4 images, 0 tables → page_2.md
  ✓ Page   3 — 6 images, 0 tables → page_3.md
  ✓ Page   4 — 1 images, 0 tables → page_4.md
  ✓ Page   5 — 2 images, 0 tables → page_5.md


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


  ✓ Page   6 — 2 images, 1 tables → page_6.md


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


  ✓ Page   7 — 1 images, 1 tables → page_7.md

Extraction complete: 3 tables, 18 images.


## Cell 12 – Assemble full document + write metadata.json

In [22]:
# ── Full document Markdown ─────────────────────────────────────────────────
# Concatenate all page files in order into a single Markdown document.
# Useful as a RAG ingestion artefact or for LLM summarisation.

divider = "\n\n---\n\n"   # horizontal rule between pages

full_doc_md = divider.join(
    page_md for _, page_md in sorted(all_pages_content, key=lambda t: t[0])
)

# Prepend a title block with document metadata
header = (
    f"# {pdf_path.stem}\n\n"
    f"*Source:* `{pdf_path.name}`  \n"
    f"*Pages:* {len(all_pages_content)}  \n"
    f"*Tables extracted:* {total_tables}  \n"
    f"*Figures extracted:* {total_images}\n\n"
)

full_md_path = doc_output_dir / f"{pdf_path.stem}_full.md"
full_md_path.write_text(header + full_doc_md, encoding="utf-8")
print(f"Full document Markdown: {full_md_path.resolve()}")
print(f"Total characters: {len(full_doc_md):,}")

# ── metadata.json ─────────────────────────────────────────────────────────
# Acts as a manifest for the extracted document:
#   - Lists all page files for easy iteration by downstream consumers
#   - Records processing timestamp for cache invalidation
#   - Stores element counts for quick quality checks

metadata = {
    "file": pdf_path.name,
    "processed": datetime.now().isoformat(),
    "tool": "Docling Bounded Notebook",
    "pages": metadata_pages,
    "total_images": total_images,
    "total_tables": total_tables
}

metadata_path = doc_output_dir / "metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata JSON: {metadata_path.resolve()}")

Full document Markdown: /Users/akellaprudhvi/mystuff/Course/GenAI-Course-Modules/Module_4_notebooks/layout_identification_extraction/extracted_docs_bounded/AI-Enablers-Adopters-research-report/AI-Enablers-Adopters-research-report_full.md
Total characters: 53,505
Metadata JSON: /Users/akellaprudhvi/mystuff/Course/GenAI-Course-Modules/Module_4_notebooks/layout_identification_extraction/extracted_docs_bounded/AI-Enablers-Adopters-research-report/metadata.json


## Cell 13 – Sanity check: preview first page boundary tags

In [23]:
# Print a summary and show the first 2000 chars of page 1 to verify
# boundary tags are being emitted correctly.

print("=" * 65)
print("EXTRACTION SUMMARY")
print("=" * 65)
print(f"PDF          : {pdf_path.name}")
print(f"Pages        : {len(all_pages_content)}")
print(f"Tables found : {total_tables}")
print(f"Figures found: {total_images}")
print(f"Output root  : {doc_output_dir.resolve()}")
print()

# Preview first page
if all_pages_content:
    first_page_no, first_page_md = all_pages_content[0]
    print(f"── Page {first_page_no} preview (first 2000 chars) ──")
    print(first_page_md[:2000])
    print()

# Count boundary tags in the full document as a quick quality check
import re
starts = len(re.findall(r"<!-- BOUNDARY_START", full_doc_md))
ends   = len(re.findall(r"<!-- BOUNDARY_END",   full_doc_md))
print(f"Boundary tags — START: {starts}, END: {ends}")
print(f"Tag balance: {'✓ balanced' if starts == ends else '✗ MISMATCH — check for truncated elements'}")

EXTRACTION SUMMARY
PDF          : AI-Enablers-Adopters-research-report.pdf
Pages        : 7
Tables found : 3
Figures found: 18
Output root  : /Users/akellaprudhvi/mystuff/Course/GenAI-Course-Modules/Module_4_notebooks/layout_identification_extraction/extracted_docs_bounded/AI-Enablers-Adopters-research-report

── Page 1 preview (first 2000 chars) ──
# Page 1


<!-- BOUNDARY_START type="header" id="p1_header_1" page="1" level="1" breadcrumbs="Thematics" -->
## Thematics
<!-- BOUNDARY_END type="header" id="p1_header_1" -->

<!-- BOUNDARY_START type="header" id="p1_header_2" page="1" level="1" breadcrumbs="Uncovering Alpha in AI's Rate of Change" -->
## Uncovering Alpha in AI's Rate of Change
<!-- BOUNDARY_END type="header" id="p1_header_2" -->

<!-- BOUNDARY_START type="paragraph" id="p1_text_1" page="1" char_count="290" word_count="50" breadcrumbs="Uncovering Alpha in AI's Rate of Change" -->
More than two years since ChatGPT's launch, we remain in the early innings of AI's diffusion. T